#### 0. LangChain에서 도구(tool) 활용 방법

LLM이 **외부 함수(도구)** 를 사용할 수 있게 연결하는 가장 기본적인 흐름을 단계별로 익힙니다.

**핵심 개념: LLM은 도구를 직접 실행하지 않습니다.**
LLM은 "이 도구를 이 인자로 호출해 주세요"라는 **요청(`tool_calls`)만 만들고**, 실제 실행은 우리 코드가 하며, 실행 결과를 다시 LLM에게 전달합니다.

```text
질문 ─▶ LLM ─▶ tool_calls(요청) ─▶ [우리 코드가 도구 실행] ─▶ ToolMessage(결과) ─▶ LLM ─▶ 최종 답변
```

| 단계 | 내용 | 핵심 코드 |
|---|---|---|
| 1 | 도구 정의 | `@tool` |
| 2 | LLM에 도구 연결 | `llm.bind_tools([...])` |
| 3 | LLM의 도구 호출 "요청" 확인 | `ai_message.tool_calls` |
| 4 | 도구 실행 후 결과를 LLM에게 전달 | `tool.invoke(tool_call)` → `ToolMessage` |

> 4단계의 수동 과정을 자동으로 반복해 주는 것이 에이전트(Agent)입니다. (`20-3Agents.ipynb` 참고)

In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정 (API 키를 코드에 직접 쓰지 않기 위함)
load_dotenv()

# 제공자별 API 키: Groq 모델은 GROQ_API_KEY, ChatUpstage는 UPSTAGE_API_KEY를 사용
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

In [ ]:
# [대안] Groq 모델을 쓰는 경우 (OpenAI 호환 API라서 ChatOpenAI로 연결)
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(
#     base_url="https://api.groq.com/openai/v1",
#     model="meta-llama/llama-4-scout-17b-16e-instruct",
#     #model="moonshotai/kimi-k2-instruct-0905",
#     temperature=0
# )

# 도구 호출(tool calling)을 지원하는 모델이어야 bind_tools를 사용할 수 있다.
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
print(llm.model_name)

### 1단계. 도구 정의

- [tool decorator](https://python.langchain.com/docs/how_to/custom_tools/#tool-decorator)를 사용하면 일반 함수를 쉽게 도구로 만들 수 있습니다
- **함수 이름**은 도구 이름, **docstring**은 도구 설명, **타입 힌트**는 인자 스키마가 됩니다
- LLM은 이 설명을 보고 어떤 도구를 쓸지 판단하므로 docstring을 명확하게 작성해야 합니다

In [ ]:
from langchain_core.tools import tool

# @tool: 일반 함수를 LLM이 호출할 수 있는 "도구"로 만든다.
#  - 함수 이름          -> 도구 이름
#  - docstring          -> 도구 설명 (LLM이 이 설명을 보고 사용할 도구를 결정한다)
#  - 타입 힌트(a: int)  -> 인자 스키마 (LLM이 만들 인자의 타입)
@tool
def add(a: int, b: int) -> int:
    """더하기 숫자 a와 b를 더합니다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """곱하기 숫자 a와 b를 곱합니다."""
    return a * b

# 일반 함수가 아니라 StructuredTool 객체로 바뀐다.
print(type(add))
print(type(multiply))

In [ ]:
# LLM에게 전달되는 도구 정보: 이름, 설명(docstring), 인자 스키마
print("이름:", add.name)
print("설명:", add.description)
print("인자:", add.args)

In [ ]:
# @tool 데코레이터로 선언된 함수는 일반 함수처럼 직접 호출할 수 없다.
# 직접 호출하면 TypeError: 'StructuredTool' object is not callable
# add(10,20)

# 도구는 invoke()에 인자를 dict로 전달해 실행한다.
add.invoke({"a": 10, "b": 20})

### 2단계. LLM에 도구 연결하기

- 먼저 도구 없이 LLM만 호출했을 때와, 도구를 연결했을 때의 차이를 알아봅니다

In [ ]:
# 도구 없이 LLM만 호출: 간단한 계산은 LLM이 스스로 답한다.
# (도구는 정확한 계산, 실시간 정보 조회, 외부 시스템 연동처럼 LLM이 직접 할 수 없는 일에 필요하다)
query = '3 곱하기 5는?'
llm_result = llm.invoke(query)  # AIMessage 반환

print(llm_result.content)       # 답변 텍스트

- 도구 리스트는 LLM에 해당하는 `BaseModel` 클래스의 `bind_tools` 메서드를 통해 전달합니다
- `bind_tools`는 원본 LLM을 바꾸지 않고, **도구가 연결된 새로운 Runnable**을 반환합니다

In [ ]:
# bind_tools: LLM에 도구 목록(이름/설명/인자 스키마)을 연결한다.
# 이후 호출할 때마다 도구 정보가 LLM에 함께 전달된다.
llm_with_tools = llm.bind_tools([add, multiply])

print(type(llm_with_tools))  # RunnableBinding: "LLM + 도구"가 묶인 객체
print(llm_with_tools)        # 출력이 길다: 내부에 원본 LLM(bound)과 연결된 도구 정보(kwargs)가 들어 있다

### 3단계. LLM의 도구 호출 "요청" 확인하기

- 도구가 필요하다고 판단하면 LLM은 답변 대신 **도구 호출 요청**을 `AIMessage`의 **`tool_calls`** 속성에 담아 반환합니다
- 이때 도구는 **아직 실행되지 않았습니다.** 실행은 4단계에서 우리 코드가 합니다
- `tool_calls`는 `additional_kwargs`가 아니라 `AIMessage`의 **별도 속성**입니다

In [ ]:
# [참고] 이전에 실행했을 때의 AIMessage 출력 예시 (모델: solar-pro2)
# 도구가 필요한 질문이면 답변 대신 맨 아래 tool_calls=[...] 에 호출 요청이 채워진다.
# (content에 설명문이 들어갈지 비어 있을지는 모델마다 다르다)
"""
AIMessage(
content='[질문 "3 곱하기 5는?"에 답하기 위해 곱셈 연산이 직접적으로 필요합니다. 
         제공된 함수 중 \'multiply\' 함수가 유일하게 필요한 계산을 수행할 수 있으므로 호출합니다.]', 
additional_kwargs={'refusal': None}, 
response_metadata={
    'token_usage': {'completion_tokens': 53, 'prompt_tokens': 578, 'total_tokens': 631, 
                    'completion_tokens_details': {'accepted_prediction_tokens': 0, 
                                                  'audio_tokens': 0, 'reasoning_tokens': 0, 
                                                  'rejected_prediction_tokens': 0}, 
                     'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 432}}, 
                     'model_provider': 'openai', 'model_name': 'solar-pro2-251215', 
                     'system_fingerprint': None, 'id': '810b1e59-3322-4980-9ecc-16daf42fe59d', 
                     'finish_reason': 'tool_calls', 'logprobs': None}, 
                     id='lc_run--019be075-bde0-7323-885a-3075464b3746-0', 
                     tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 
                                  'id': 'chatcmpl-tool-63bddbf5f5ad4f179884c1a9258e5c7c', 'type': 'tool_call'}], 
                    invalid_tool_calls=[], 
                    usage_metadata={'input_tokens': 578, 'output_tokens': 53, 'total_tokens': 631, 
                                    'input_token_details': {'audio': 0, 'cache_read': 432}, 
                                    'output_token_details': {'audio': 0, 'reasoning': 0}}
)
"""

In [ ]:
# 도구가 연결된 LLM 호출: 도구가 필요한 질문이면 답변 대신 "도구 호출 요청"을 반환한다.
tool_result = llm_with_tools.invoke(query)

tool_result  # content는 비어 있을 수 있고, tool_calls에 요청 정보가 들어 있다

In [ ]:
from pprint import pprint

# [참고] AIMessage의 모든 속성 확인: tool_calls 외에 토큰 사용량 같은 부가 정보(response_metadata)가 많다
pprint(vars(tool_result))

In [ ]:
# tool_calls: LLM이 요청한 도구 호출 목록 (여러 개일 수 있다)
#  - name: 호출할 도구 이름 / args: LLM이 만든 인자 / id: 요청 식별자 (결과를 돌려줄 때 짝을 맞추는 데 사용)
tool_result.tool_calls

- 질문에 맞는 도구를 LLM이 스스로 선택합니다
- 도구가 필요 없는 질문이면 `tool_calls`는 빈 리스트입니다

In [ ]:
# 질문에 맞는 도구를 LLM이 스스로 선택한다 (곱하기 -> multiply, 더하기 -> add)
query = "10 더하기 5는 얼마인가요?"
tool_result = llm_with_tools.invoke(query)

tool_result.tool_calls

In [ ]:
# 도구가 필요 없는 질문이면 tool_calls는 빈 리스트다. (이때는 content에 일반 답변이 들어 있다)
query = '안녕하세요? 오늘의 날씨는 어떤가요?'
tool_result = llm_with_tools.invoke(query)

tool_result.tool_calls

### 4단계. 도구를 실행하고 결과를 LLM에게 전달하기 (수동 구현)

도구 호출 요청을 처리하는 전체 과정을 직접 한 단계씩 수행해 봅니다. 대화 기록에는 아래 메시지가 순서대로 쌓입니다.

```text
① HumanMessage   사용자 질문
② AIMessage      LLM의 도구 호출 요청 (tool_calls)
③ ToolMessage    우리가 도구를 실행한 결과
④ AIMessage      ①②③을 모두 본 LLM의 최종 답변
```

In [ ]:
from typing import Sequence

from langchain_core.messages import AnyMessage, HumanMessage

#query = "20과 10을 나누기 결과는?"  # 나누기 도구는 없으므로, 이 경우 LLM이 어떻게 반응하는지 확인해 볼 수 있다
query = "20과 10을 곱한 결과는?"
human_message = HumanMessage(query)  # ① 사용자 질문을 메시지 객체로 만든다
# 대화 기록(message_list): 이후 AIMessage, ToolMessage를 순서대로 이어 붙여 LLM에 다시 전달한다
message_list: Sequence[AnyMessage] = [human_message]

print(message_list)

**① → ② LLM 호출**

- `tool_calls`를 가진 `AIMessage`의 형태를 기억하기
- 이 메시지를 대화 기록(`message_list`)에 그대로 추가해야 합니다

In [ ]:
# ② LLM 호출 -> 도구 호출 요청이 담긴 AIMessage를 받는다
# llm_with_tools 는 LLM과 Tool이 연동된 RunnableBinding 객체
ai_message = llm_with_tools.invoke(message_list)
ai_message

In [ ]:
# 요청 내용 확인: 호출할 도구 이름(multiply)과 인자(a, b)가 담겨 있다
ai_message.tool_calls

In [ ]:
from pprint import pprint

# 도구 호출 요청이 담긴 AIMessage를 대화 기록에 추가한다.
# (다음 호출 때 LLM이 "내가 어떤 요청을 했는지" 알아야 하므로 반드시 필요하다)
message_list.append(ai_message)

pprint(message_list)

**③ 도구 실행**

- LLM은 도구를 실행하지 않으므로, `tool_calls`의 내용으로 **우리가 도구를 직접 실행**합니다

In [ ]:
# [위 셀과 동일] 실행할 도구 호출 요청을 다시 확인
ai_message.tool_calls

In [ ]:
# ③ 도구 실행: tool_calls 항목(dict)을 invoke()에 그대로 전달하면 결과가 ToolMessage로 반환된다.
#  - args만 전달하면 계산 결과값(예: 200)만 반환되고,
#  - tool_call 전체(name, args, id)를 전달하면 tool_call_id가 포함된 ToolMessage가 반환된다.
tool_message = multiply.invoke(ai_message.tool_calls[0])

print(tool_message)  # content: 계산 결과 / tool_call_id: 요청의 id와 동일

**③ → ④ 실행 결과를 LLM에게 돌려주기**

- 실행 결과는 `ToolMessage`로 대화 기록에 추가해 LLM에게 전달합니다
- 에이전트는 이 과정(요청 확인 → 도구 실행 → 결과 전달)을 자동으로 반복해 줍니다

In [ ]:
# ③ 도구 실행 결과(ToolMessage)를 대화 기록에 추가한다
message_list.append(tool_message)

pprint(message_list)

In [ ]:
# ④ [질문, 도구 호출 요청, 도구 결과]를 함께 전달하면 LLM이 결과를 반영한 최종 답변을 생성한다
tool_result = llm_with_tools.invoke(message_list)

pprint(tool_result)  # content에 최종 답변이 들어 있다

- `message_list`의 순서를 기억하기: `HumanMessage` → `AIMessage(tool_calls)` → `ToolMessage`
- 이 순서와 `tool_call_id` 짝이 어긋나면 오류가 나거나 엉뚱한 답이 나올 수 있습니다

In [ ]:
# 최종 대화 기록: HumanMessage -> AIMessage(tool_calls) -> ToolMessage 순서
message_list

#### 정리

| 메시지 | 만든 주체 | 역할 |
|---|---|---|
| `HumanMessage` | 사용자 | 질문 |
| `AIMessage` (`tool_calls`) | LLM | 도구 호출 **요청** (실행하지 않음) |
| `ToolMessage` | 우리 코드 | 도구 **실행 결과** (`tool_call_id`로 요청과 연결) |
| `AIMessage` (최종) | LLM | 결과를 반영한 답변 |

- 도구 연결의 핵심은 **`@tool`로 정의 → `bind_tools`로 연결 → `tool_calls` 확인 → 실행 결과를 `ToolMessage`로 반환**입니다
- 이 반복을 자동으로 처리해 주는 것이 에이전트입니다 (`20-3Agents.ipynb`)